# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-nancyzakria-hash/flyrank-intern-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import duckdb
from google.colab import userdata

# 1. Fetch HF Token from Colab Secrets into Python variable
HF_TOKEN = userdata.get('HF_TOKEN')

# 2. Connect DuckDB & Load HTTPFS Extension
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# 3. Create Hugging Face Secret
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

# 4. Test Query on Dataset Path
rel = "hf://datasets/FlyRank/internship-warehouse"
result = con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')").df()

print("HF Token loaded & DuckDB connected successfully!")
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HF Token loaded & DuckDB connected successfully!
   count_star()
0      78835655


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Inspect all available columns across the dataset using union_by_name
inspect_df = con.execute("""
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet', union_by_name=true, hive_partitioning=true)
    LIMIT 5
""").df()

print("Available columns:")
print(list(inspect_df.columns))

Available columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Verification Queries & Leakage Trap
# Below are three verification queries on the mid-panel month (`2026-03`):
# 1. Grain verification (confirming zero duplicate rows).
# 2. Row count and date span.
# 3. Availability check using `is_available IS TRUE`.# Define target month and dataset path

# Define target month and specific fact table path
TARGET_MONTH = "2026-03"
DATA_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Query 1: Verify Grain Uniqueness (Checking for duplicate rows)
query_grain = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
FROM read_parquet('{DATA_PATH}', hive_partitioning=true)
WHERE month = '{TARGET_MONTH}' OR strftime(report_date, '%Y-%m') = '{TARGET_MONTH}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING row_count > 1;
"""
duplicates_df = con.execute(query_grain).df()
print(f"Fact 1 - Duplicate grain rows: {len(duplicates_df)}")

# Query 2: Row Count and Date Span
query_span = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM read_parquet('{DATA_PATH}', hive_partitioning=true)
WHERE month = '{TARGET_MONTH}' OR strftime(report_date, '%Y-%m') = '{TARGET_MONTH}';
"""
span_df = con.execute(query_span).df()
print("\nFact 2 - Row Count & Date Span:")
print(span_df)

# Query 3: Availability Check using IS TRUE
query_availability = f"""
SELECT COUNT(*) as available_rows
FROM read_parquet('{DATA_PATH}', hive_partitioning=true)
WHERE (month = '{TARGET_MONTH}' OR strftime(report_date, '%Y-%m') = '{TARGET_MONTH}')
  AND (gsc_data_available IS TRUE OR ga4_data_available IS TRUE);
"""
availability_df = con.execute(query_availability).df()
print("\nFact 3 - Available Rows (is_available IS TRUE):")
print(availability_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 1 - Duplicate grain rows: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Fact 2 - Row Count & Date Span:
   total_rows start_date   end_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Fact 3 - Available Rows (is_available IS TRUE):
   available_rows
0         3660680


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Data Limitation: Google Search Console thresholding suppresses low-volume search queries for privacy. Consequently, our dataset misses long-tail queries, causing an unobserved query bias.
# Feature availability explanations:
# - hist_impressions: knowable at the decision moment because logged prior to evaluation window.
# - hist_ctr: knowable at the decision moment because computed from historical search logs.
# - avg_position: knowable at the decision moment because recorded in GSC history prior to query event.
# scroll_events :

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Load mid-panel data
df = con.execute(f"""
    SELECT
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        scroll_events,
        (COALESCE(ga4_engaged_sessions, 0) > 0) AS label,
        CAST((COALESCE(ga4_engaged_sessions, 0) > 0) AS DOUBLE) AS leaked_feature
    FROM read_parquet('{DATA_PATH}', hive_partitioning=true)
    WHERE month = '{TARGET_MONTH}' AND (gsc_data_available IS TRUE OR ga4_data_available IS TRUE)
""").df()


leaked_features = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'scroll_events', 'leaked_feature']
honest_features = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'scroll_events']

y = df['label']

# Leakage Experiment (The Trap)
clf_leak = DecisionTreeClassifier(max_depth=3)
clf_leak.fit(df[leaked_features], y)
score_leak = roc_auc_score(y, clf_leak.predict_proba(df[leaked_features])[:, 1])
print(f"Score with Leaked Feature (The Trap): {score_leak:.4f}")

# Honest Evaluation (Leak Removed)
clf_honest = DecisionTreeClassifier(max_depth=3)
clf_honest.fit(df[honest_features], y)
score_honest = roc_auc_score(y, clf_honest.predict_proba(df[honest_features])[:, 1])
print(f"Honest Score without Leaked Feature: {score_honest:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Score with Leaked Feature (The Trap): 1.0000
Honest Score without Leaked Feature: 0.9870


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.